In [109]:
import pandas as pd

In [110]:
df = pd.read_csv("DataCoSupplyChainDataset.csv")

C:\Users\THISLAPTOP\AppData\Local\Temp\ipykernel_7288\3124520526.py:1: DtypeWarning: Columns (27,39,45,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("DataCoSupplyChainDataset.csv")


In [111]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360.0,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360.0,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360.0,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360.0,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360.0,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [112]:
df.isnull().sum()

Type                                  0
Days for shipping (real)              0
Days for shipment (scheduled)         0
Benefit per order                     0
Sales per customer                    0
Delivery Status                       0
Late_delivery_risk                    0
Category Id                           0
Category Name                         0
Customer City                         0
Customer Country                      0
Customer Email                        0
Customer Fname                        0
Customer Id                           0
Customer Lname                        8
Customer Password                     0
Customer Segment                      0
Customer State                        0
Customer Street                       0
Customer Zipcode                      3
Department Id                         0
Department Name                       0
Latitude                              0
Longitude                             0
Market                                0


In [113]:
df.isnull().sum().sum()

np.int64(346798)

In [114]:
df.shape

(180519, 53)

## Concatinate (marge Fname + Lname)

In [115]:
# fillna('') se jahan NaN hai wahan empty text aa jayega
df['Customer Full Name'] = df['Customer Fname'].fillna('') + ' ' + df['Customer Lname'].fillna('')

# Ab check karein, NaN khatam ho jayenge
print(df['Customer Full Name'].head())


0       Cally Holloway
1           Irene Luna
2    Gillian Maldonado
3            Tana Tate
4       Orli Hendricks
Name: Customer Full Name, dtype: object


## Remove Unneeded column 

In [116]:
# 2. Remove sensitive columns (Customer Email and Customer Password)
cols_to_remove = ['Customer Email', 'Customer Password','Order Zipcode','Product Card Id','Product Category Id','Product Description','Customer Fname','Customer Lname','Customer Zipcode']
df = df.drop(columns=cols_to_remove, errors='ignore')
print(f"Removed columns: {cols_to_remove}")

Removed columns: ['Customer Email', 'Customer Password', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Customer Fname', 'Customer Lname', 'Customer Zipcode']


In [117]:
df.shape

(180519, 45)

## null Show is here > 0

In [118]:
null_summary = df.isnull().sum()
print("\nNull Values per column:")
print(null_summary[null_summary > 0]) # Only show columns that have nulls


Null Values per column:
Order State                   3004
Order Status                  4581
shipping date (DateOrders)    3004
Shipping Mode                 7585
dtype: int64


## Order State handle missing value

In [119]:
if df['Order State'].isnull().any():
    os_mode = df['Order State'].mode()[0]
    df['Order State'] = df['Order State'].fillna(os_mode)
    print(f"Filled nulls in 'Order State' with mode: {os_mode}")

Filled nulls in 'Order State' with mode: Inglaterra


In [120]:
print(df[['Customer Full Name', 'Order State']].isnull().sum())

Customer Full Name    0
Order State           0
dtype: int64


# Fix Order Status and Order Region
# Since 'Order Region' has 'Region?Status', we split it

In [121]:
def fix_region_and_status(row):
    region_val = str(row['Order Region'])
    if '?' in region_val:
        split_val = region_val.split('?')
        return split_val[0], split_val[1]
    return row['Order Region'], row['Order Status']
df[['Order Region', 'Order Status']] = df.apply(
    lambda x: pd.Series(fix_region_and_status(x)), axis=1
)

### Logic for Restoration

In [122]:
def reconstruct_data(row):
    region = str(row['Order Region']).strip()
    state = str(row['Order State']).strip()
    status = str(row['Order Status']).strip().lower() # The 'code' like s, ia, n, ov
    
    # Valid Status List (to identify when State has a status)
    valid_statuses = ['COMPLETE', 'PENDING', 'CLOSED', 'PROCESSING', 
                      'SUSPECTED_FRAUD', 'ON_HOLD', 'CANCELED', 'PENDING_PAYMENT']
    
    # Check for the codes (s, ia, n, ov) in Status
    if status in ['s', 'ia', 'n', 'ov']:
        # Fix Region
        new_region = region + status
        
        # Fix Status by taking it from State
        if state.upper() in valid_statuses:
            new_status = state.upper()
        else:
            new_status = None # Will be filled by mode later
            
        return pd.Series([new_region, new_status])
    
    return pd.Series([row['Order Region'], row['Order Status']])
# Apply the logic
df[['Order Region', 'Order Status']] = df.apply(reconstruct_data, axis=1)
# 2. Final Clean-up (Remove '?' and fill nulls)
df['Order Region'] = df['Order Region'].str.replace('?', '', regex=False)
df['Order Status'] = df['Order Status'].fillna(df['Order Status'].mode()[0])
print("Restoration Complete!")
df[['Order Region', 'Order Status', 'Order State']].head(20)

Restoration Complete!


,Order Region,Order Status,Order State
0,Southeast Asia,COMPLETE,Java Occidental
1,South Asia,PENDING,Rajast?
2,South Asia,CLOSED,Rajast?
3,Oceania,COMPLETE,Queensland
4,Oceania,PENDING_PAYMENT,Queensland
5,Oceania,CANCELED,Queensland
6,Eastern Asia,COMPLETE,Guangdong
7,Eastern Asia,PROCESSING,Guangdong
8,Eastern Asia,CLOSED,Guangdong
9,Eastern Asia,CLOSED,Guangdong


# REMOVE ALL '?' FROM STATE AND REGION (Fixing your issue)

In [29]:
df['Order State'] = df['Order State'].str.replace('?', '', regex=False).str.strip()
df['Order Region'] = df['Order Region'].str.replace('?', '', regex=False).str.strip()

# Fix Nulls in 'Order State' using the Mode

In [123]:
if df['Order State'].isnull().any():
    os_mode = df['Order State'].mode()[0]
    df['Order State'] = df['Order State'].fillna(os_mode)

In [124]:
# 6. Final Verification
print("Final Null Check:")
print(df[['Customer Full Name', 'Order State', 'Order Status', 'Order Region']].isnull().sum())
print("\nUnique Order Statuses:")
print(df['Order Status'].unique())
df.head()

Final Null Check:
Customer Full Name    0
Order State           0
Order Status          0
Order Region          0
dtype: int64

Unique Order Statuses:
['COMPLETE' 'PENDING' 'CLOSED' 'PENDING_PAYMENT' 'CANCELED' 'PROCESSING'
 'SUSPECTED_FRAUD' 'ON_HOLD' 'PAYMENT_REVIEW' ' Paulo' ''
 ' Grande del Norte' ' de Janeiro']


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Region,Order State,Order Status,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode,Customer Full Name
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,Southeast Asia,Java Occidental,COMPLETE,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class,Cally Holloway
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,South Asia,Rajast?,PENDING,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class,Irene Luna
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,South Asia,Rajast?,CLOSED,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class,Gillian Maldonado
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,Oceania,Queensland,COMPLETE,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class,Tana Tate
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,Oceania,Queensland,PENDING_PAYMENT,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class,Orli Hendricks


In [125]:
if df['Order State'].isnull().any():
    state_mode = df['Order State'].mode()[0]
    df['Order State'] = df['Order State'].fillna(state_mode)
    print(f"Filled Order State nulls with mode: {state_mode}")

In [126]:
df.shape

(180519, 45)

In [127]:
null_summary = df.isnull().sum()
print("\nNull Values per column:")
print(null_summary[null_summary > 0]) # Only show columns that have nulls


Null Values per column:
shipping date (DateOrders)    3004
Shipping Mode                 7585
dtype: int64


In [128]:
# Clean data ko nayi csv file mein save karein
df.to_csv('1Cleaned_DataCoSupplyChainDataset.csv', index=False, encoding='latin1')
print("1Cleaned CSV file saved successfully!")

1Cleaned CSV file saved successfully!
